# Claude Extensions
## Vom halluzinierenden Chat zur kontrollierten Agenten-Schleife



In [1]:
from IPython.display import display, HTML
from pathlib import Path
import html
import json
import textwrap
import uuid

CSS = r"""
<style>
:root{
  --ink:#0f172a;--muted:#475569;--line:#94a3b8;--panel:#ffffff;--soft:#f8fafc;
  --blue:#2563eb;--purple:#7e22ce;--green:#15803d;--red:#b91c1c;--amber:#a16207;
  --cyan:#0891b2;--orange:#ea580c;--pink:#be185d;
}
.ce-stage{font-family:Inter,Segoe UI,Arial,sans-serif;color:var(--ink);margin:12px 0 28px;line-height:1.45}
.ce-stage *{box-sizing:border-box}.ce-stage h2,.ce-stage h3,.ce-stage h4,.ce-stage p{color:var(--ink)!important}
.ce-hero{position:relative;overflow:hidden;border-radius:24px;padding:30px 34px;color:white;background:linear-gradient(120deg,#1d4ed8,#7e22ce 58%,#be185d);box-shadow:0 18px 42px rgba(30,41,59,.22)}
.ce-hero h2{font-size:34px;line-height:1.12;margin:0 0 12px;color:white!important}.ce-hero p{font-size:18px;line-height:1.55;margin:0;color:white!important;max-width:920px}
.ce-card{background:#fff;border:1px solid #cbd5e1;border-left:7px solid var(--blue);border-radius:18px;padding:18px 21px;margin:14px 0;box-shadow:0 5px 16px rgba(15,23,42,.08)}
.ce-blue{background:#eff6ff;border-left-color:var(--blue)}.ce-green{background:#f0fdf4;border-left-color:var(--green)}.ce-red{background:#fef2f2;border-left-color:var(--red)}.ce-amber{background:#fffbeb;border-left-color:var(--amber)}.ce-purple{background:#faf5ff;border-left-color:var(--purple)}.ce-neutral{background:#f8fafc;border-left-color:#64748b}
.ce-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:14px;margin:16px 0}.ce-mini{background:white;border:1px solid #cbd5e1;border-radius:15px;padding:16px;min-height:108px;box-shadow:0 4px 12px rgba(15,23,42,.06)}
.ce-mini .label{font-size:12px;font-weight:850;letter-spacing:.08em;text-transform:uppercase;color:var(--muted)!important;margin-bottom:8px}.ce-mini .value{font-size:17px;font-weight:720;line-height:1.4;color:var(--ink)!important}
.ce-badge{display:inline-block;padding:6px 10px;border-radius:999px;font-size:13px;font-weight:850;margin:4px 5px 4px 0;border:1px solid transparent}.b-blue{background:#dbeafe;color:#1e3a8a}.b-green{background:#dcfce7;color:#14532d}.b-red{background:#fee2e2;color:#7f1d1d}.b-amber{background:#fef3c7;color:#78350f}.b-purple{background:#f3e8ff;color:#581c87}.b-gray{background:#e2e8f0;color:#334155}
.ce-btn{display:inline-flex;align-items:center;gap:7px;cursor:pointer;border:0;border-radius:12px;background:linear-gradient(90deg,#2563eb,#7e22ce);color:white!important;padding:10px 15px;font-weight:850;box-shadow:0 6px 16px rgba(37,99,235,.22);margin:8px 6px 4px 0}.ce-btn:hover{transform:translateY(-1px)}
.ce-callout{font-size:21px;font-weight:900;text-align:center;padding:17px;border-radius:15px;background:linear-gradient(90deg,#dcfce7,#dbeafe,#f3e8ff);color:#0f172a!important;box-shadow:0 7px 22px rgba(37,99,235,.12)}
.ce-code{background:#0f172a;color:#f8fafc!important;border-radius:15px;padding:17px 19px;white-space:pre-wrap;overflow-x:auto;font:14px/1.52 ui-monospace,SFMono-Regular,Consolas,monospace;box-shadow:inset 0 0 0 1px #334155}.ce-code *{color:#f8fafc!important}

/* Mock chat */
.chat-shell{max-width:980px;margin:18px auto;border:1px solid #cbd5e1;border-radius:22px;overflow:hidden;background:white;box-shadow:0 18px 45px rgba(15,23,42,.14)}
.chat-top{height:58px;background:#f8fafc;border-bottom:1px solid #e2e8f0;display:flex;align-items:center;justify-content:space-between;padding:0 18px}.chat-brand{display:flex;align-items:center;gap:10px;font-weight:850}.chat-logo{width:31px;height:31px;border-radius:10px;background:linear-gradient(135deg,#2563eb,#7e22ce);display:grid;place-items:center;color:white!important}.chat-dots{display:flex;gap:6px}.chat-dot{width:9px;height:9px;border-radius:50%;background:#cbd5e1}
.chat-body{min-height:430px;padding:26px 28px;background:linear-gradient(#fff,#f8fafc)}.chat-row{display:flex;margin:18px 0;opacity:0;transform:translateY(14px)}.chat-row.user{justify-content:flex-end}.chat-row.assistant{justify-content:flex-start}.chat-bubble{max-width:72%;padding:14px 17px;border-radius:18px;font-size:17px;line-height:1.5;box-shadow:0 4px 12px rgba(15,23,42,.08)}.user .chat-bubble{background:#2563eb;color:white!important;border-bottom-right-radius:5px}.assistant .chat-bubble{background:#fff;border:1px solid #cbd5e1;color:#0f172a!important;border-bottom-left-radius:5px}.chat-label{font-size:11px;text-transform:uppercase;letter-spacing:.08em;font-weight:850;margin-bottom:5px;opacity:.76}.typing{display:flex;gap:5px;align-items:center}.typing i{width:8px;height:8px;border-radius:50%;background:#64748b;animation:typing 1.1s infinite}.typing i:nth-child(2){animation-delay:.15s}.typing i:nth-child(3){animation-delay:.3s}
.chat-eval{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:10px;margin-top:18px}.eval-pill{opacity:0;transform:translateY(10px);background:#fff;border:1px solid #fecaca;border-left:5px solid #dc2626;border-radius:13px;padding:12px 13px;font-weight:780;color:#7f1d1d!important}.eval-pill.ok{border-color:#bbf7d0;border-left-color:#16a34a;color:#14532d!important}.eval-pill.info{border-color:#bfdbfe;border-left-color:#2563eb;color:#1e3a8a!important}
.chat-play .user-q{animation:reveal .45s .25s forwards}.chat-play .typing-row{animation:reveal .45s 1.0s forwards}.chat-play .answer-row{animation:reveal .5s 2.15s forwards}.chat-play .typing-row{animation:reveal .45s 1s forwards,hide .2s 2.0s forwards}.chat-play .e1{animation:reveal .4s 3.15s forwards}.chat-play .e2{animation:reveal .4s 3.65s forwards}.chat-play .e3{animation:reveal .4s 4.15s forwards}.chat-play .e4{animation:reveal .4s 4.65s forwards}.chat-play .e5{animation:reveal .4s 5.15s forwards}
.chat-play.skill-chat .s1{animation:reveal .4s 1.5s forwards}.chat-play.skill-chat .s2{animation:reveal .4s 2.0s forwards}.chat-play.skill-chat .s3{animation:reveal .4s 2.5s forwards}.chat-play.skill-chat .s4{animation:reveal .4s 3.0s forwards}.chat-play.skill-chat .answer-row{animation:reveal .5s 3.75s forwards}.chat-play.skill-chat .typing-row{animation:reveal .45s 1s forwards,hide .2s 3.55s forwards}

/* Sequence diagram */
.ce-sequence{border:1px solid #cbd5e1;border-radius:20px;background:white;box-shadow:0 12px 32px rgba(15,23,42,.10);padding:15px;overflow-x:auto}.seq-title{display:flex;align-items:flex-start;justify-content:space-between;gap:15px;padding:4px 7px 12px}.seq-title h3{margin:0;font-size:22px}.seq-title p{margin:4px 0 0;color:#475569!important}.seq-canvas{min-width:1080px}.ce-sequence svg{width:100%;height:auto;display:block}.seq-step,.seq-note{opacity:0}.seq-play .seq-step{animation:seqShow .45s forwards}.seq-play .seq-note{animation:seqShow .42s forwards}.seq-line{stroke-dasharray:800;stroke-dashoffset:800}.seq-play .seq-line{animation:drawLine .6s forwards}.seq-controls{padding:8px 7px 2px}.seq-log{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:10px;padding:8px 7px 2px}.seq-note{background:#f8fafc;border:1px solid #cbd5e1;border-radius:13px;padding:11px 12px}.seq-note .n{font-size:12px;font-weight:900;color:#2563eb!important;text-transform:uppercase;letter-spacing:.06em}.seq-note .t{font-size:14px;color:#0f172a!important;margin-top:5px}

/* Proportionale 80%-Darstellung nur für ausgewählte Sequenzdiagramme */
.ce-sequence.seq-compact .seq-canvas{
  min-width:0;
  width:80%;
  max-width:1024px;
  margin:0 auto;
}
.ce-sequence.seq-compact .seq-canvas svg{
  width:100%;
  height:auto;
  display:block;
}

/* Auf sehr schmalen Ausgabeflächen darf die Grafik den verfügbaren Platz nutzen. */
@media (max-width:760px){
  .ce-sequence.seq-compact .seq-canvas{
    width:100%;
    max-width:none;
  }
}

/* Skill editor */
.skill-shell{display:grid;grid-template-columns:300px 1fr;border:1px solid #334155;border-radius:18px;overflow:hidden;background:#0f172a;box-shadow:0 18px 40px rgba(15,23,42,.22);min-height:440px}.skill-tree{background:#111827;color:#e5e7eb!important;padding:20px;font:14px/1.7 ui-monospace,Consolas,monospace;border-right:1px solid #334155}.skill-editor{padding:0;background:#0b1220}.editor-tab{height:48px;display:flex;align-items:center;padding:0 18px;background:#111827;border-bottom:1px solid #334155;color:#e5e7eb!important;font:14px ui-monospace,Consolas,monospace}.editor-code{padding:20px 23px;color:#e5e7eb!important;font:14px/1.62 ui-monospace,Consolas,monospace}.code-line{opacity:0;transform:translateX(-8px);white-space:pre-wrap}.skill-play .l1{animation:reveal .35s .25s forwards}.skill-play .l2{animation:reveal .35s .55s forwards}.skill-play .l3{animation:reveal .35s .85s forwards}.skill-play .l4{animation:reveal .35s 1.15s forwards}.skill-play .l5{animation:reveal .35s 1.45s forwards}.skill-play .l6{animation:reveal .35s 1.75s forwards}.skill-play .l7{animation:reveal .35s 2.05s forwards}.skill-play .l8{animation:reveal .35s 2.35s forwards}.skill-play .l9{animation:reveal .35s 2.65s forwards}.skill-play .l10{animation:reveal .35s 2.95s forwards}.skill-play .l11{animation:reveal .35s 3.25s forwards}.skill-play .l12{animation:reveal .35s 3.55s forwards}.skill-play .save-dot{animation:savePulse 1s 4s 3}

/* Layer map and plugin */
.layer-map{display:grid;gap:10px;margin:18px 0}.layer{display:grid;grid-template-columns:180px 1fr;align-items:center;border-radius:15px;padding:14px 16px;border:1px solid #cbd5e1;background:white;opacity:0;transform:translateX(-14px)}.layer strong{font-size:16px}.layer-map.play .layer:nth-child(1){animation:reveal .45s .2s forwards}.layer-map.play .layer:nth-child(2){animation:reveal .45s .55s forwards}.layer-map.play .layer:nth-child(3){animation:reveal .45s .9s forwards}.layer-map.play .layer:nth-child(4){animation:reveal .45s 1.25s forwards}.layer-map.play .layer:nth-child(5){animation:reveal .45s 1.6s forwards}.layer-map.play .layer:nth-child(6){animation:reveal .45s 1.95s forwards}.layer-map.play .layer:nth-child(7){animation:reveal .45s 2.3s forwards}
.plugin-box{position:relative;min-height:330px;border:2px dashed #7e22ce;border-radius:22px;background:#faf5ff;padding:65px 25px 25px;margin-top:18px}.plugin-title{position:absolute;top:16px;left:22px;font-size:23px;font-weight:900;color:#581c87!important}.plugin-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:13px}.plugin-part{opacity:0;transform:scale(.85);background:white;border:1px solid #d8b4fe;border-radius:15px;padding:17px;text-align:center;font-weight:850}.plugin-box.play .plugin-part:nth-child(1){animation:pop .45s .2s forwards}.plugin-box.play .plugin-part:nth-child(2){animation:pop .45s .55s forwards}.plugin-box.play .plugin-part:nth-child(3){animation:pop .45s .9s forwards}.plugin-box.play .plugin-part:nth-child(4){animation:pop .45s 1.25s forwards}.plugin-box.play .plugin-part:nth-child(5){animation:pop .45s 1.6s forwards}.plugin-box.play .plugin-part:nth-child(6){animation:pop .45s 1.95s forwards}

@keyframes reveal{to{opacity:1;transform:translate(0,0)}}@keyframes hide{to{opacity:0;visibility:hidden}}@keyframes typing{0%,60%,100%{transform:translateY(0)}30%{transform:translateY(-5px)}}@keyframes seqShow{to{opacity:1}}@keyframes drawLine{to{stroke-dashoffset:0}}@keyframes pop{to{opacity:1;transform:scale(1)}}@keyframes savePulse{0%,100%{box-shadow:0 0 0 0 rgba(34,197,94,0)}50%{box-shadow:0 0 0 10px rgba(34,197,94,.25)}}
@media(max-width:800px){.skill-shell{grid-template-columns:1fr}.skill-tree{border-right:0;border-bottom:1px solid #334155}.chat-bubble{max-width:90%}.layer{grid-template-columns:1fr;gap:5px}}
@media(prefers-reduced-motion:reduce){.ce-stage *{animation-duration:.01ms!important;animation-delay:0ms!important;transition:none!important}}
</style>
"""
display(HTML(CSS))


def render(body: str) -> None:
    display(HTML(f'<div class="ce-stage">{body}</div>'))


def esc(value: object) -> str:
    return html.escape(str(value))


def chat_hallucination() -> str:
    return r"""
    <div class="chat-shell chat-play">
      <div class="chat-top"><div class="chat-brand"><div class="chat-logo">AI</div>Standard-Chat</div><div class="chat-dots"><span class="chat-dot"></span><span class="chat-dot"></span><span class="chat-dot"></span></div></div>
      <div class="chat-body">
        <div class="chat-row user user-q"><div class="chat-bubble"><div class="chat-label">User</div>Wann und wo findet die North Sea AI Reliability Conference 2028 statt?</div></div>
        <div class="chat-row assistant typing-row"><div class="chat-bubble"><div class="typing"><i></i><i></i><i></i></div></div></div>
        <div class="chat-row assistant answer-row"><div class="chat-bubble"><div class="chat-label">Assistant</div>Die Konferenz findet am <strong>12. und 13. September 2028</strong> im <strong>CCH Hamburg</strong> statt.</div></div>
        <div class="chat-eval">
          <div class="eval-pill e1">✕ Keine Quelle genannt</div>
          <div class="eval-pill e2">✕ Keine Internetsuche</div>
          <div class="eval-pill e3">✕ Kein verifizierter Kontext</div>
          <div class="eval-pill e4">✕ Kein definierter Fallback</div>
          <div class="eval-pill e5">⚠ Plausibel, aber erfunden</div>
        </div>
        <div style="margin-top:12px"><button class="ce-btn" onclick="const w=this.closest('.chat-shell');w.classList.remove('chat-play');void w.offsetWidth;w.classList.add('chat-play');">↻ Animation wiederholen</button></div>
      </div>
    </div>"""


def chat_with_skill() -> str:
    return r"""
    <div class="chat-shell chat-play skill-chat">
      <div class="chat-top"><div class="chat-brand"><div class="chat-logo">AI</div>Claude mit Skill</div><div><span class="ce-badge b-green">Skill aktiv</span></div></div>
      <div class="chat-body">
        <div class="chat-row user user-q"><div class="chat-bubble"><div class="chat-label">User</div>Wann und wo findet die North Sea AI Reliability Conference 2028 statt?</div></div>
        <div class="chat-row assistant typing-row"><div class="chat-bubble"><div class="typing"><i></i><i></i><i></i></div></div></div>
        <div class="chat-eval">
          <div class="eval-pill info s1">✓ Passender Skill erkannt</div>
          <div class="eval-pill info s2">✓ Quellenprüfung vorgeschrieben</div>
          <div class="eval-pill info s3">✓ Keine Quelle verfügbar</div>
          <div class="eval-pill ok s4">✓ Fallback-Regel anwenden</div>
        </div>
        <div class="chat-row assistant answer-row"><div class="chat-bubble"><div class="chat-label">Assistant</div><strong>Ich habe keine Quellen gefunden.</strong></div></div>
        <div style="margin-top:12px"><button class="ce-btn" onclick="const w=this.closest('.chat-shell');w.classList.remove('chat-play');void w.offsetWidth;w.classList.add('chat-play');">↻ Animation wiederholen</button></div>
      </div>
    </div>"""


def _wrap_lines(text: str, width: int = 34) -> list[str]:
    return (textwrap.wrap(text, width=width) or [''])[:2]


def sequence_diagram(
    title: str,
    subtitle: str,
    actors: list[dict],
    phases: list[dict],
    steps: list[dict],
    compact: bool = False,
) -> str:
    uid = 'seq-' + uuid.uuid4().hex[:8]
    width, left, right = 1280, 190, 100
    # Die interne Geometrie bleibt immer unverändert.
    # Ausgewählte Diagramme werden ausschließlich proportional per CSS skaliert.
    top, row_h, bottom = 78, 54, 28
    gap = (width - left - right) / max(len(actors) - 1, 1)
    xs = [left + i * gap for i in range(len(actors))]
    actor_index = {a['name']: i for i, a in enumerate(actors)}
    h = top + 32 + len(steps) * row_h + bottom
    phase_colors = ['#ecfccb','#dbeafe','#ffedd5','#f3e8ff','#cffafe','#fee2e2']
    svg = [f'<svg viewBox="0 0 {width} {h}" role="img" aria-label="{esc(title)}">']
    svg.append(f'<defs><marker id="arrow-{uid}" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#334155"/></marker></defs>')
    for pi, phase in enumerate(phases):
        y1 = top + 18 + phase['start'] * row_h
        y2 = top + 18 + (phase['end'] + 1) * row_h
        color = phase.get('color', phase_colors[pi % len(phase_colors)])
        svg.append(f'<rect x="100" y="{y1}" width="{width-120}" height="{y2-y1}" rx="4" fill="{color}" opacity="0.76"/>')
        cy = (y1 + y2) / 2
        svg.append(f'<circle cx="34" cy="{cy}" r="18" fill="white" stroke="#cbd5e1" stroke-width="2"/>')
        svg.append(f'<text x="34" y="{cy+5}" text-anchor="middle" font-size="14" font-weight="800" fill="#475569">{pi+1}</text>')
        svg.append(f'<text x="62" y="{cy+5}" text-anchor="start" font-size="15" font-weight="800" fill="#475569">{esc(phase["name"])}</text>')
    for i, actor in enumerate(actors):
        x, color = xs[i], actor.get('color','#2563eb')
        w = max(105, min(180, 20 + len(actor['label']) * 8))
        svg.append(f'<rect x="{x-w/2}" y="16" width="{w}" height="36" rx="7" fill="{color}"/>')
        svg.append(f'<text x="{x}" y="39" text-anchor="middle" font-size="13" font-weight="850" fill="white">{esc(actor["label"])}</text>')
        svg.append(f'<line x1="{x}" y1="52" x2="{x}" y2="{h-14}" stroke="{color}" stroke-width="2" opacity="0.5"/>')
    notes = []
    for si, step in enumerate(steps):
        y = top + 46 + si * row_h
        x1, x2 = xs[actor_index[step['from']]], xs[actor_index[step['to']]]
        delay, color = si * 0.72, step.get('color','#334155')
        if x1 == x2:
            path = f'M {x1} {y} h 48 v 25 h -48'
            svg.append(f'<g class="seq-step" style="animation-delay:{delay}s"><path class="seq-line" style="animation-delay:{delay}s" d="{path}" fill="none" stroke="{color}" stroke-width="2.4" marker-end="url(#arrow-{uid})"/></g>')
            tx = x1 + 58
        else:
            direction = 1 if x2 > x1 else -1
            start_x, end_x = x1 + 8*direction, x2 - 10*direction
            svg.append(f'<g class="seq-step" style="animation-delay:{delay}s"><line class="seq-line" style="animation-delay:{delay}s" x1="{start_x}" y1="{y}" x2="{end_x}" y2="{y}" stroke="{color}" stroke-width="2.4" marker-end="url(#arrow-{uid})"/></g>')
            tx = (x1 + x2) / 2
        for li, line in enumerate(_wrap_lines(step['label'], 30)):
            svg.append(f'<text class="seq-step" style="animation-delay:{delay}s" x="{tx}" y="{y-8+li*14}" text-anchor="middle" font-size="12" font-weight="760" fill="#0f172a">{esc(line)}</text>')
        svg.append(f'<circle class="seq-step" style="animation-delay:{delay}s" cx="{x1}" cy="{y}" r="5" fill="{color}"/>')
        notes.append(f'<div class="seq-note" style="animation-delay:{delay+0.18}s"><div class="n">Schritt {si+1} · {esc(step.get("type","message"))}</div><div class="t">{esc(step.get("description", step["label"]))}</div></div>')
    svg.append('</svg>')
    compact_class = " seq-compact" if compact else ""
    return f"""
    <div class="ce-sequence seq-play{compact_class}">
      <div class="seq-title"><div><h3>{esc(title)}</h3><p>{esc(subtitle)}</p></div><span class="ce-badge b-purple">Ask · Choose · Do · Tell</span></div>
      <div class="seq-canvas">{''.join(svg)}</div>
      <div class="seq-controls"><button class="ce-btn" onclick="const w=this.closest('.ce-sequence');w.classList.remove('seq-play');void w.offsetWidth;w.classList.add('seq-play');">↻ Animation wiederholen</button></div>
      <div class="seq-log">{''.join(notes)}</div>
    </div>"""


def intro_hallucination_panel() -> str:
    return """
    <div class="ce-hero"><h2>Eine konkrete Frage – aber keine überprüfbaren Daten</h2><p>Der Chat wirkt vertraut. Genau deshalb fällt eine flüssig formulierte Halluzination leicht durch.</p></div>
    """ + chat_hallucination() + """
    """


def transition_panel() -> str:
    return """
    <div class="ce-callout">Was ist zwischen der Nutzerfrage und dieser Antwort technisch passiert?</div>
    <div class="ce-grid">
      <div class="ce-mini"><div class="label">Input</div><div class="value">Eine User-Nachricht ohne Quelle</div></div>
      <div class="ce-mini"><div class="label">Modell</div><div class="value">Erzeugt plausible Textfortsetzung</div></div>
      <div class="ce-mini"><div class="label">Output</div><div class="value">Flüssige, aber unbelegte Antwort</div></div>
    </div>"""


def message_types_panel() -> str:
    return r"""
    <div class="ce-grid">
      <div class="ce-card ce-blue"><h3><code>user</code></h3><p>Die eigentliche Anfrage oder eine Information des Nutzers.</p><div class="ce-code">{"role":"user","content":"Wann findet die Konferenz statt?"}</div></div>
      <div class="ce-card ce-purple"><h3><code>assistant</code></h3><p>Text, Tool-Entscheidung oder sonstiger Inhalt, den das Modell zurückgibt.</p><div class="ce-code">{"role":"assistant","content":"Die Konferenz findet ..."}</div></div>
      <div class="ce-card ce-neutral"><h3>System und Kontext</h3><p>Regeln, verfügbare Tools, geladene Skills und bisheriger Nachrichtenverlauf bestimmen den Rahmen der Anfrage.</p></div>
    </div>
    <div class="ce-card ce-amber"><h3>Warum war die Antwort riskant?</h3><p>Im Ablauf gab es weder eine Quellenanforderung noch ein Suchwerkzeug oder eine definierte Reaktion auf fehlende Evidenz.</p></div>"""


def skill_editor_html(skill_content: str, skill_path: Path) -> str:
    lines = skill_content.splitlines()
    line_html = []
    for i, line in enumerate(lines[:12], start=1):
        safe = esc(line) if line else '&nbsp;'
        line_html.append(f'<div class="code-line l{i}"><span style="color:#64748b">{i:02d}</span>  {safe}</div>')
    return f"""
    <div class="skill-shell skill-play">
      <div class="skill-tree">📁 demo_project<br>└─ 📁 .claude<br>&nbsp;&nbsp;&nbsp;└─ 📁 skills<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└─ 📁 reliable-event-search<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└─ 📄 <strong>SKILL.md</strong></div>
      <div class="skill-editor"><div class="editor-tab"><span class="save-dot" style="display:inline-block;width:9px;height:9px;border-radius:50%;background:#22c55e;margin-right:9px"></span>SKILL.md</div><div class="editor-code">{''.join(line_html)}</div></div>
    </div>
    <button class="ce-btn" onclick="const w=this.previousElementSibling;w.classList.remove('skill-play');void w.offsetWidth;w.classList.add('skill-play');">↻ Skill-Anlage wiederholen</button>
    <div class="ce-card ce-green"><h3>✓ Datei wirklich geschrieben</h3><p><code>{esc(skill_path)}</code></p></div>"""


def skill_result_panel() -> str:
    return chat_with_skill() + """
    <div class="ce-card ce-green"><h3>Was hat sich geändert?</h3><p>Nicht das Weltwissen des Modells, sondern sein vorgeschriebenes Verhalten bei fehlender Evidenz.</p></div>"""


def before_after_panel() -> str:
    return r"""
    <div class="ce-grid">
      <div class="ce-card ce-red"><h3>Ohne Skill</h3><p><strong>„12.–13. September 2028, CCH Hamburg“</strong></p><p>Konkrete, aber unbelegte Aussage.</p></div>
      <div class="ce-card ce-green"><h3>Mit Skill</h3><p><strong>„Ich habe keine Quellen gefunden.“</strong></p><p>Keine unbelegten Fakten; definierter Fallback.</p></div>
    </div>
    <div class="ce-callout">Ein Skill reduziert das Risiko – belastbare Fakten benötigen zusätzlich eine Datenquelle oder ein Tool.</div>"""


def tool_blocks_panel(tool_use: dict, tool_result: dict) -> str:
    return f"""
    <div class="ce-grid">
      <div class="ce-card ce-purple"><h3><code>tool_use</code></h3><p>Claude beschreibt die gewünschte Aktion und ihre Argumente.</p><div class="ce-code">{esc(json.dumps(tool_use, ensure_ascii=False, indent=2))}</div></div>
      <div class="ce-card ce-blue"><h3><code>tool_result</code></h3><p>Die Anwendung liefert das ausgeführte Ergebnis zurück.</p><div class="ce-code">{esc(json.dumps(tool_result, ensure_ascii=False, indent=2))}</div></div>
    </div>
    <div class="ce-card ce-amber"><h3>Wichtig</h3><p>Der <code>tool_use</code>-Block ist keine ausgeführte Funktion. Er ist die strukturierte Aufforderung an den Client, die Funktion auszuführen.</p></div>"""


def client_mcp_compare_panel() -> str:
    return """
    <div class="ce-grid">
      <div class="ce-card ce-blue"><h3>Direktes Client-Tool</h3><p>Die Application kennt und implementiert die Funktion unmittelbar.</p><span class="ce-badge b-blue">individuelle Integration</span></div>
      <div class="ce-card ce-green"><h3>MCP-Tool</h3><p>Ein MCP-Server stellt Tools und Datenquellen über eine standardisierte Schnittstelle bereit.</p><span class="ce-badge b-green">wiederverwendbare Integration</span></div>
    </div>"""


def plugin_panel() -> str:
    return r"""
    <div class="plugin-box play"><div class="plugin-title">Claude Plugin</div><div class="plugin-grid">
      <div class="plugin-part">skills/</div><div class="plugin-part">agents/</div><div class="plugin-part">hooks/</div><div class="plugin-part">commands/</div><div class="plugin-part">.mcp.json</div><div class="plugin-part">plugin.json</div>
    </div></div>
    <button class="ce-btn" onclick="const w=this.previousElementSibling;w.classList.remove('play');void w.offsetWidth;w.classList.add('play');">↻ Verpackung wiederholen</button>
    <div class="ce-card ce-purple"><h3>Einordnung</h3><p>Das Plugin bündelt Komponenten. Die Entscheidungslogik bleibt bei Skill-Matching, Modell, Host, Hooks und Agenten.</p></div>"""


def layer_map_panel() -> str:
    return r"""
    <div class="layer-map play">
      <div class="layer" style="border-left:7px solid #15803d"><strong>Kontext</strong><span><b>Skills</b> liefern Regeln, Wissen und Workflows.</span></div>
      <div class="layer" style="border-left:7px solid #2563eb"><strong>Nachrichten</strong><span><b>user, assistant, tool_use, tool_result</b> bilden den Dialogzustand.</span></div>
      <div class="layer" style="border-left:7px solid #ea580c"><strong>Ausführung</strong><span><b>Tools</b> führen Funktionen aus oder beschaffen Daten.</span></div>
      <div class="layer" style="border-left:7px solid #0891b2"><strong>Integration</strong><span><b>MCP-Server</b> stellen Tools und Datenquellen standardisiert bereit.</span></div>
      <div class="layer" style="border-left:7px solid #b91c1c"><strong>Kontrolle</strong><span><b>Hooks</b> prüfen und erzwingen Regeln an Lifecycle-Ereignissen.</span></div>
      <div class="layer" style="border-left:7px solid #7e22ce"><strong>Orchestrierung</strong><span><b>Agenten und Subagenten</b> koordinieren mehrere Schritte und Delegation.</span></div>
      <div class="layer" style="border-left:7px solid #be185d"><strong>Distribution</strong><span><b>Plugins</b> bündeln die Erweiterungen für Teams und Organisationen.</span></div>
    </div>
    <button class="ce-btn" onclick="const w=this.previousElementSibling;w.classList.remove('play');void w.offsetWidth;w.classList.add('play');">↻ Ebenen wiederholen</button>"""


def final_panel() -> str:
    return r"""
    <div class="ce-grid">
      <div class="ce-card ce-red"><h3>Vorher</h3><p>User-Frage → Modell → plausible Antwort</p><p><span class="ce-badge b-red">0 Quellen</span><span class="ce-badge b-red">kein Fallback</span></p></div>
      <div class="ce-card ce-green"><h3>Nachher</h3><p>User-Frage → Skill-Regeln → Tool/MCP → Quellenresultat → kontrollierte Antwort</p><p><span class="ce-badge b-green">keine unbelegten Fakten</span><span class="ce-badge b-green">definierter Fallback</span></p></div>
    </div>
    <div class="ce-callout">„Ich habe keine Quellen gefunden.“ ist hier keine schwache Antwort, sondern das korrekte Ergebnis eines kontrollierten Systems.</div>"""

render('<div class="ce-card ce-green"><h3>✓ Präsentationssystem geladen</h3><p>Mock-Chats, Swimlane-Sequenzdiagramme und Offline-Demos sind bereit.</p></div>')

## Agenda

1. Halluzination im gemockten Standard-Chat
2. Technischer Nachrichtenfluss hinter dem Chat
3. Skill als echte `SKILL.md` anlegen
4. Gleiche Frage mit kontrolliertem Fallback
5. Tool Use im Diagramm
6. MCP, Hooks, Agenten, Subagenten und Plugins einordnen
7. Abschluss: Fragen, Anmerkungen?


# 1 · Halluzination als Einstieg

In [2]:
render(intro_hallucination_panel())

In [ ]:
render(transition_panel())

# 2 · Der normale Nachrichtenfluss

Zuerst betrachten wir nur den Basischat: **User → Anwendung → Claude → Anwendung → User**.

In [ ]:
actors = [
    {'name':'user','label':'User','color':'#6d28d9'},
    {'name':'app','label':'Application','color':'#0284c7'},
    {'name':'claude','label':'Claude API','color':'#ea580c'},
]
phases = [
    {'name':'Ask','start':0,'end':1},
    {'name':'Choose','start':2,'end':2},
    {'name':'Tell','start':3,'end':4},
]
steps = [
    {'from':'user','to':'app','label':'Terminfrage eingeben','type':'user input','description':'Der Nutzer formuliert die Frage im Chatfenster.'},
    {'from':'app','to':'claude','label':'messages: user','type':'API request','description':'Die Anwendung sendet den vorhandenen Nachrichtenverlauf an Claude.'},
    {'from':'claude','to':'claude','label':'Antwort generieren','type':'model inference','description':'Claude verarbeitet nur den vorhandenen Kontext; eine externe Quelle wurde nicht abgefragt.'},
    {'from':'claude','to':'app','label':'assistant message','type':'API response','description':'Die Modellantwort wird als Assistant-Inhalt an die Anwendung zurückgegeben.'},
    {'from':'app','to':'user','label':'Antwort anzeigen','type':'UI rendering','description':'Das Chatfenster rendert die Antwort für den Nutzer.'},
]
render(sequence_diagram('Basischat ohne externe Daten','Eine Modellantwort kann nur auf dem mitgegebenen Kontext und den verfügbaren Fähigkeiten beruhen.',actors,phases,steps))

## 2.1 Technische Umsetzunge - Blick hinter die Kulissen

In [ ]:
render(message_types_panel())

# 3 · Einen Skill als echte Datei anlegen

Der Skill ergänzt wiederverwendbare Regeln. Er ist **kein Suchwerkzeug** und erzeugt keine externen Daten.  ---> Claude Demo

In [ ]:
render(skill_editor_html(SKILL_CONTENT, SKILL_PATH))

In [ ]:
actors = [
    {'name':'user','label':'User','color':'#6d28d9'},
    {'name':'host','label':'Claude Code Host','color':'#0284c7'},
    {'name':'skill','label':'SKILL.md','color':'#15803d'},
    {'name':'claude','label':'Claude','color':'#ea580c'},
]
phases = [
    {'name':'Ask','start':0,'end':0},
    {'name':'Enrich','start':1,'end':2},
    {'name':'Choose','start':3,'end':3},
    {'name':'Tell','start':4,'end':5},
]
steps = [
    {'from':'user','to':'host','label':'Terminanfrage','type':'user message','description':'Die gleiche Frage erreicht den Host.'},
    {'from':'host','to':'skill','label':'Beschreibung passt','type':'skill discovery','description':'Der Host erkennt den Skill als relevant und lädt seine Anweisungen.'},
    {'from':'skill','to':'host','label':'Quellenregeln laden','type':'context enrichment','description':'Die Skill-Datei liefert Regeln; sie führt selbst keine Aktion aus.'},
    {'from':'host','to':'claude','label':'Frage + Skill-Regeln','type':'model request','description':'Claude erhält die Frage zusammen mit dem kontrollierten Vorgehen.'},
    {'from':'claude','to':'host','label':'Fallback-Antwort','type':'assistant response','description':'Ohne verfügbare Quelle werden keine Terminangaben erfunden.'},
    {'from':'host','to':'user','label':'Ehrliche Antwort','type':'UI rendering','description':'Der Nutzer erhält den definierten Fallback.'},
]
render(sequence_diagram('Wo der Skill wirkt','SKILL.md ist eine geladene Ressource im Kontext – kein eigenständiger Agent und kein Tool.',actors,phases,steps))

# 4 · Gleiche Frage, kontrollierte Antwort

Jetzt läuft dieselbe Chat-Szene noch einmal – diesmal mit den Regeln des Skills.

In [ ]:
render(skill_result_panel())

In [ ]:
render(before_after_panel())

# 5 · Tool Use: Ask → Choose → Do → Tell

Nun ergänzen wir den Nachrichtenfluss um eine echte ausführbare Funktion. Das Diagramm orientiert sich an klassischen Function-Calling-Sequenzdiagrammen.

In [ ]:
actors = [
    {'name':'user','label':'User','color':'#6d28d9'},
    {'name':'app','label':'Application','color':'#0284c7'},
    {'name':'claude','label':'Claude API','color':'#ea580c'},
    {'name':'fn','label':'Search Function','color':'#be185d'},
]
phases = [
    {'name':'Ask','start':0,'end':1},
    {'name':'Choose','start':2,'end':2},
    {'name':'Do','start':3,'end':4},
    {'name':'Tell','start':5,'end':8},
]
steps = [
    {'from':'user','to':'app','label':'Terminfrage','type':'user input','description':'Der Nutzer stellt die Frage.'},
    {'from':'app','to':'claude','label':'messages + tools','type':'API request','description':'Die Anwendung übergibt Nachrichten und Tool-Beschreibungen.'},
    {'from':'claude','to':'app','label':'tool_use + arguments','type':'tool decision','description':'Claude fordert search_event mit strukturierten Argumenten an.'},
    {'from':'app','to':'fn','label':'search_event(...)','type':'function call','description':'Die Anwendung führt die tatsächliche Python-Funktion aus.'},
    {'from':'fn','to':'app','label':'0 Treffer','type':'function result','description':'Die Suche liefert im Offline-Demoindex keine Quellen.'},
    {'from':'app','to':'app','label':'messages aktualisieren','type':'client state','description':'Das Tool-Ergebnis wird dem Nachrichtenverlauf hinzugefügt.'},
    {'from':'app','to':'claude','label':'tool_result','type':'API request 2','description':'Die Anwendung sendet das Ergebnis mit passender tool_use_id zurück.'},
    {'from':'claude','to':'app','label':'final text','type':'API response 2','description':'Claude formuliert auf Basis des Ergebnisses die finale Antwort.'},
    {'from':'app','to':'user','label':'Antwort anzeigen','type':'UI rendering','description':'Das Chatfenster zeigt den kontrollierten Fallback.'},
]
render(sequence_diagram('Claude Tool Use – Client führt die Funktion aus','Das Modell wählt das Tool; die Application pausiert, führt es aus und sendet tool_result zurück.',actors,phases,steps,compact=True))

# 5.1 Blick hinter die Kulissen

In [ ]:
tool_use = {
    'type':'tool_use','id':'toolu_demo_01','name':'search_event',
    'input':{'event_name':'North Sea AI Reliability Conference 2028'}
}
tool_result = {'type':'tool_result','tool_use_id':'toolu_demo_01','content':[]}
render(tool_blocks_panel(tool_use, tool_result))

# 6 · MCP: standardisierte Tool-Bereitstellung

MCP ersetzt nicht die Tool-Entscheidung des Modells. Es standardisiert, wie externe Werkzeuge und Datenquellen bereitgestellt und angesprochen werden.

In [ ]:
actors = [
    {'name':'user','label':'User','color':'#6d28d9'},
    {'name':'host','label':'Host / Application','color':'#0284c7'},
    {'name':'claude','label':'Claude','color':'#ea580c'},
    {'name':'mcp','label':'MCP Server','color':'#0891b2'},
    {'name':'source','label':'External Source','color':'#15803d'},
]
phases = [
    {'name':'Ask','start':0,'end':1},
    {'name':'Choose','start':2,'end':2},
    {'name':'Do','start':3,'end':6},
    {'name':'Tell','start':7,'end':9},
]
steps = [
    {'from':'user','to':'host','label':'Terminfrage','type':'user input','description':'Die Anfrage beginnt wie zuvor.'},
    {'from':'host','to':'claude','label':'messages + MCP tools','type':'model request','description':'Der Host exponiert die vom MCP-Server bereitgestellten Tools.'},
    {'from':'claude','to':'host','label':'Tool auswählen','type':'tool decision','description':'Claude entscheidet anhand der Toolbeschreibung.'},
    {'from':'host','to':'mcp','label':'MCP tool call','type':'protocol call','description':'Der Host leitet den strukturierten Aufruf über MCP weiter.'},
    {'from':'mcp','to':'source','label':'Quelle abfragen','type':'external access','description':'Der MCP-Server kapselt die systemspezifische Integration.'},
    {'from':'source','to':'mcp','label':'keine Treffer','type':'source result','description':'Die externe Quelle liefert keine verifizierbare Information.'},
    {'from':'mcp','to':'host','label':'standardisiertes Ergebnis','type':'MCP result','description':'Der Host erhält ein strukturiertes Tool-Ergebnis.'},
    {'from':'host','to':'claude','label':'tool_result','type':'model continuation','description':'Das Ergebnis wird in den Modellkontext zurückgeführt.'},
    {'from':'claude','to':'host','label':'final text','type':'assistant response','description':'Claude formuliert den Fallback.'},
    {'from':'host','to':'user','label':'Antwort anzeigen','type':'UI rendering','description':'Der Nutzer erhält die finale Antwort.'},
]
render(sequence_diagram('MCP-Tool-Flow','Der MCP-Server kapselt Tooldefinition, Protokoll und Zugriff auf das externe System.',actors,phases,steps,compact=True))

# 6.1 Technische Anmerkung

In [ ]:
render(client_mcp_compare_panel())

# 7 · Hook: Kontrolle im Ausführungspfad

Ein Hook ist kein normaler Nachrichtentyp und kein frei gewähltes Tool. Er wird an einem definierten Lifecycle-Ereignis automatisch ausgeführt.

In [ ]:
actors = [
    {'name':'claude','label':'Claude','color':'#ea580c'},
    {'name':'host','label':'Claude Code Host','color':'#0284c7'},
    {'name':'hook','label':'PreToolUse Hook','color':'#b91c1c'},
    {'name':'tool','label':'Bash Tool','color':'#64748b'},
]
phases = [
    {'name':'Choose','start':0,'end':0},
    {'name':'Validate','start':1,'end':2},
    {'name':'Do','start':3,'end':3},
    {'name':'Tell','start':4,'end':4},
]
steps = [
    {'from':'claude','to':'host','label':'tool_use: Bash','type':'tool decision','description':'Claude fordert einen Bash-Aufruf an.'},
    {'from':'host','to':'hook','label':'Befehl prüfen','type':'automatic hook','description':'Der Host startet den PreToolUse-Hook automatisch.'},
    {'from':'hook','to':'host','label':'BLOCKED','type':'policy result','description':'Der Hook blockiert den riskanten Befehl deterministisch.'},
    {'from':'host','to':'tool','label':'nicht ausgeführt','type':'execution prevented','description':'Das Bash-Tool wird nicht aufgerufen.'},
    {'from':'host','to':'claude','label':'Blockier-Ergebnis','type':'feedback','description':'Claude erhält die Information, dass der Aufruf nicht erlaubt war.'},
]
render(sequence_diagram('Hook als Sicherheitsbarriere','Beispiel: rm -rf ./demo-output wird lediglich simuliert und niemals ausgeführt.',actors,phases,steps))

# 8 · Agent: derselbe Flow in einer Schleife

Ein Agent ist nicht ein weiterer Nachrichtenblock. Er orchestriert wiederholte Modell- und Tool-Aufrufe, bis ein Ziel erreicht oder eine Abbruchbedingung erfüllt ist.

In [ ]:
actors = [
    {'name':'user','label':'User','color':'#6d28d9'},
    {'name':'agent','label':'Agent Loop','color':'#7e22ce'},
    {'name':'claude','label':'Claude','color':'#ea580c'},
    {'name':'tool','label':'Search Tool','color':'#15803d'},
]
phases = [
    {'name':'Goal','start':0,'end':0},
    {'name':'Plan','start':1,'end':2},
    {'name':'Act','start':3,'end':4},
    {'name':'Observe','start':5,'end':6},
    {'name':'Tell','start':7,'end':8},
]
steps = [
    {'from':'user','to':'agent','label':'Ziel übergeben','type':'task','description':'Der Nutzer formuliert das gewünschte Endergebnis.'},
    {'from':'agent','to':'claude','label':'messages + tools','type':'iteration 1','description':'Der Agent startet einen Modellschritt.'},
    {'from':'claude','to':'agent','label':'tool_use','type':'plan','description':'Claude entscheidet über den nächsten Arbeitsschritt.'},
    {'from':'agent','to':'tool','label':'Suche ausführen','type':'action','description':'Der Agent führt das gewählte Tool aus.'},
    {'from':'tool','to':'agent','label':'Ergebnis','type':'observation','description':'Das Tool liefert eine Beobachtung zurück.'},
    {'from':'agent','to':'claude','label':'tool_result','type':'iteration 2','description':'Die Beobachtung wird in den nächsten Modellschritt gegeben.'},
    {'from':'claude','to':'agent','label':'fertig oder weiter?','type':'decision','description':'Claude bewertet, ob ein weiterer Schritt nötig ist.'},
    {'from':'claude','to':'agent','label':'final text','type':'completion','description':'Die Schleife endet mit einer finalen Antwort.'},
    {'from':'agent','to':'user','label':'Ergebnis liefern','type':'response','description':'Der Agent gibt das konsolidierte Ergebnis zurück.'},
]
render(sequence_diagram('Agenten-Schleife','Planen → Handeln → Beobachten → erneut planen – mit klarer Endbedingung.',actors,phases,steps,compact=True))

# 9 · Subagent: Delegation mit eigenem Kontext

Der Hauptagent delegiert eine spezialisierte Teilaufgabe. Der Subagent arbeitet isoliert und liefert ein kompaktes Ergebnis zurück.

In [ ]:
actors = [
    {'name':'user','label':'User','color':'#6d28d9'},
    {'name':'main','label':'Main Agent','color':'#7e22ce'},
    {'name':'sub','label':'Research Subagent','color':'#be185d'},
    {'name':'tool','label':'Search Tool','color':'#15803d'},
]
phases = [
    {'name':'Delegate','start':0,'end':1},
    {'name':'Work','start':2,'end':3},
    {'name':'Return','start':4,'end':4},
    {'name':'Tell','start':5,'end':5},
]
steps = [
    {'from':'user','to':'main','label':'Termin prüfen','type':'task','description':'Der Hauptagent erhält die übergeordnete Aufgabe.'},
    {'from':'main','to':'sub','label':'Offizielle Quelle suchen','type':'delegation','description':'Eine spezialisierte Rechercheaufgabe wird delegiert.'},
    {'from':'sub','to':'tool','label':'Suchanfrage','type':'subagent action','description':'Der Subagent nutzt nur die für ihn freigegebenen Tools.'},
    {'from':'tool','to':'sub','label':'keine Treffer','type':'observation','description':'Die Suche liefert keine Quelle.'},
    {'from':'sub','to':'main','label':'kompaktes Ergebnis','type':'return','description':'Nur das relevante Resultat fließt in den Hauptkontext zurück.'},
    {'from':'main','to':'user','label':'Fallback-Antwort','type':'response','description':'Der Hauptagent formuliert die finale Antwort.'},
]
render(sequence_diagram('Subagenten-Delegation','Eigener Kontext, eigene Instruktionen und begrenzte Tools – Rückgabe eines kompakten Ergebnisses.',actors,phases,steps))

# 10 · Wo passt welche Erweiterung hin?

In [ ]:
render(layer_map_panel())

# 12 · Abschluss: dieselbe Frage, ein kontrollierter Prozess

In [ ]:
render(final_panel())

# Vielen Dank für die Aufmerksamkeit!

## Anregungen, Fragen, Meinungen ?

